# PHYT1D — Module 01: Base ODE Structure (Dalla Man Skeleton)

Implements the 10-state glucose-insulin ODE system:
- Subcutaneous insulin absorption (Schiavon 2018)
- Oral glucose absorption (Dalla Man 2006)
- Glucose-insulin kinetics (Dalla Man 2014 / Bergman 1979)
- Hypoglycaemia risk function ρ(G)

**References:** Dalla Man 2014 [8]; Schiavon 2018 [9]; Bergman 1979 [14]; Dalla Man 2006 [10]


In [ ]:
import numpy as np

# ─── Fixed population parameters (Group D) ───────────────────────────────────
FIXED = {
    # Subcutaneous insulin PK (Schiavon 2018)
    "VI"   : 0.126,    # L/kg      — volume of insulin distribution
    "ke"   : 0.127,    # min⁻¹     — insulin fractional clearance rate
    "beta" : 8.0,      # min       — SC absorption delay

    # Oral glucose absorption (Dalla Man 2006)
    "f"    : 0.90,     # —         — fraction of intestinal CHO absorbed

    # Glucose kinetics
    "VG"   : 1.45,     # dL/kg     — volume of glucose distribution
    "alpha": 7.0,      # min       — interstitial glucose delay time constant

    # Hypoglycaemia risk function (Dalla Man 2014)
    "r1"   : 1.44,     # —         — risk function shape parameter 1
    "r2"   : 0.81,     # —         — risk function shape parameter 2
    "Gth"  : 60.0,     # mg/dL     — hypoglycaemia glucose threshold

    # Non-insulin-mediated glucose uptake (Hovorka 2004)
    "Fc01"      : 0.0097,  # mmol/kg/min — baseline NIMGU (brain + erythrocytes)
    "gamma_aer" : 0.003,   # —           — aerobic muscle uptake coefficient
    "gamma_res" : 0.0015,  # —           — resistance muscle uptake coefficient

    # Exercise timing (Riddell 2017; Yardley 2013)
    "tau_ramp"  : 20.0,    # min    — ramp-up time for muscle uptake / resistance EGP
    "delta_res" : 0.003,   # —      — resistance SI reduction coefficient
}


## 1.1 Hypoglycaemia Risk Function ρ(G)

In [ ]:
def rho(G, Gb, p=FIXED):
    """
    
    Hypoglycaemia risk SCORING function (Dalla Man 2014).
    Used ONLY for clinical risk metrics (Clarke EGA, TBR scoring).
    NOT used inside the glucose ODE — X(t) acts on G unconditionally.
    ...

    Hypoglycaemia risk function from Dalla Man 2014.

    Parameters
    ----------
    G  : float — plasma glucose [mg/dL]
    Gb : float — basal plasma glucose [mg/dL]
    p  : dict  — fixed parameters

    Returns
    -------
    float — ρ(G) ≥ 0
    """
    r1, r2, Gth = p["r1"], p["r2"], p["Gth"]
    if G >= Gb:
        return 0.0
    elif G > Gth:
        return (10 ** r1) * (np.log(G) ** r2 - np.log(Gb) ** r2) ** 2
    else:
        return (10 ** r1) * (np.log(Gth) ** r2 - np.log(Gb) ** r2) ** 2


## 1.2 Subcutaneous Insulin Absorption (Schiavon 2018)

In [ ]:
def dIsc1_dt(Isc1, I_input, kd, p=FIXED):
    """
    dIsc1/dt = -kd * Isc1(t) + I(t - beta) / VI

    Isc1  [mU/kg] — non-monomeric insulin in SC depot
    I_input       — delayed insulin infusion rate [mU/kg/min]
    kd    [min⁻¹] — diffusion rate (identified per patient, Group A)
    """
    return -kd * Isc1 + I_input / p["VI"]


def dIsc2_dt(Isc1, Isc2, kd, ka2):
    """
    dIsc2/dt = kd * Isc1(t) - ka2 * Isc2(t)

    Isc2  [mU/kg] — monomeric insulin in SC tissue
    ka2   [min⁻¹] — SC absorption rate (identified per patient, Group A)
    """
    return kd * Isc1 - ka2 * Isc2


def dIp_dt(Isc2, Ip, ka2, p=FIXED):
    """
    dIp/dt = ka2 * Isc2(t) - ke * Ip(t)

    Ip [mU/L] — plasma insulin concentration
    ke [min⁻¹] — insulin clearance rate (fixed, Group D)
    """
    return ka2 * Isc2 - p["ke"] * Ip


## 1.3 Oral Glucose Absorption (Dalla Man 2006)

In [ ]:
def dQsto1_dt(Qsto1, CHO_rate, kempt):
    """
    dQsto1/dt = -kempt * Qsto1(t) + CHO(t)

    Qsto1    [mg/kg] — solid-phase stomach glucose
    CHO_rate [mg/kg/min] — meal ingestion rate at time t
    kempt    [min⁻¹] — gastric emptying rate (identified per patient, Group A)
    """
    return -kempt * Qsto1 + CHO_rate


def dQsto2_dt(Qsto1, Qsto2, kempt):
    """
    dQsto2/dt = kempt * Qsto1(t) - kempt * Qsto2(t)

    Qsto2 [mg/kg] — liquid-phase stomach glucose
    """
    return kempt * Qsto1 - kempt * Qsto2


def dQgut_dt(Qsto2, Qgut, kempt, kabs):
    """
    dQgut/dt = kempt * Qsto2(t) - kabs * Qgut(t)

    Qgut [mg/kg] — intestinal glucose
    kabs [min⁻¹] — intestinal absorption rate (identified per patient, Group A)
    """
    return kempt * Qsto2 - kabs * Qgut


def Ra(Qgut, kabs, p=FIXED):
    """
    Ra(t) = f * kabs * Qgut(t)   [mg/kg/min]

    Glucose appearance rate in plasma from gut.
    f = 0.90 (fixed, Group D) — fraction of CHO absorbed.
    """
    return p["f"] * kabs * Qgut


## 1.4 Glucose-Insulin Kinetics (Dalla Man 2014 / Bergman 1979)

In [ ]:
def dG_dt(G, X, Ra_val, SG, Gb, p=FIXED):
    """
    dG/dt = -(SG + X)*G + SG*Gb + Ra(t)/VG

    X acts unconditionally on glucose disposal (Dalla Man 2014).
    rho(G) is a separate risk-scoring function, not an ODE term.
    """
    return -(SG + X) * G + SG * Gb + Ra_val / p["VG"]


def dX_dt(X, Ip, Ipb, SI_eff, p2):
    """
    dX/dt = -p2 * [X(t) - SI_eff(t) * (Ip(t) - Ipb)]

    X    [min⁻¹] — remote insulin action (proportional to insulin effect on glucose)
    p2   [min⁻¹] — insulin action rate constant (Group A)
    Ipb  [mU/L]  — basal plasma insulin
    """
    return -p2 * (X - SI_eff * (Ip - Ipb))


def dIG_dt(IG, G, p=FIXED):
    """
    dIG/dt = -(1/alpha) * [IG(t) - G(t)]

    IG [mg/dL] — interstitial glucose (measured by CGM)
    alpha [min] — CGM lag time constant (fixed, Group D)
    """
    return -(1.0 / p["alpha"]) * (IG - G)


## 1.5 EGP — Endogenous Glucose Production

In [ ]:
def EGP_base(t, EGP0=0.0161):
    """
    Baseline endogenous glucose production.
    EGP0 [mmol/kg/min] — basal EGP (population value, ~0.0161).
    Returns constant baseline; exercise increments are added in phi module.
    """
    return EGP0


## 1.6 Quick Sanity Check

In [ ]:
# Quick sanity checks — instantaneous evaluation of each ODE function
_G0, _Gb = 120.0, 120.0          # mg/dL
_Ip0, _Ipb = 10.0, 10.0          # mU/L
_X0 = 0.0                         # min⁻¹ — at basal, X starts at zero

print("rho(120, 120)      :", rho(_G0, _Gb))                        # expect 0
print("rho( 80, 120)      :", round(rho(80.0, _Gb), 6))             # expect > 0
print("dG/dt at basal     :", round(dG_dt(_G0, _X0, _Ip0, _Ipb,
                                           0.0, 1e-4, 0.025, _Gb), 6))  # expect ≈ 0
print("dIG/dt at basal    :", dIG_dt(_G0, _G0))                     # expect 0
print("Ra(Qgut=10, kabs)  :", round(Ra(10.0, 0.057), 4))           # expect 0.0513
print("dQsto1 (meal=1)    :", dQsto1_dt(0.0, 1.0, 0.055))          # expect 1.0
print("EGP_base           :", EGP_base(0))

print("\n✓ All ODE components defined successfully.")
